In [2]:
import numpy as np
import pandas as pd
import json
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Imports successful")
print("📊 Loading DKRL next-day click prediction dataset...")


✓ Imports successful
📊 Loading DKRL next-day click prediction dataset...


In [3]:
# Load main dataset arrays
print("📥 Loading core dataset arrays...")

X_user = np.load('../data_processing/X_user_features.npy')
X_news = np.load('../data_processing/X_news_embeddings.npy') 
y = np.load('../data_processing/y_clicks.npy')

print(f"✓ User features: {X_user.shape}")
print(f"✓ News embeddings: {X_news.shape}")
print(f"✓ Click labels: {y.shape}")
print(f"✓ Total memory: ~{(X_user.nbytes + X_news.nbytes + y.nbytes) / 1024**2:.1f} MB")


📥 Loading core dataset arrays...
✓ User features: (18309, 219)
✓ News embeddings: (18309, 100)
✓ Click labels: (18309,)
✓ Total memory: ~44.7 MB


In [4]:
# Load metadata and supporting files
print("📋 Loading metadata and supporting files...")

# Load metadata
metadata_df = pd.read_csv('../data_processing/prediction_metadata.csv')

# Load dataset info
with open('../data_processing/dataset_info.json', 'r') as f:
    dataset_info = json.load(f)

# Load feature names
with open('../data_processing/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

# Load user profiles
with open('../data_processing/user_profiles.pkl', 'rb') as f:
    user_profiles = pickle.load(f)

print(f"✓ Metadata: {len(metadata_df)} records")
print(f"✓ Dataset info: {len(dataset_info)} properties")
print(f"✓ Feature names: {len(feature_names)} features")
print(f"✓ User profiles: {len(user_profiles)} users")


📋 Loading metadata and supporting files...
✓ Metadata: 18309 records
✓ Dataset info: 14 properties
✓ Feature names: 219 features
✓ User profiles: 500 users


In [5]:
# Display comprehensive dataset information
print("🎯 DKRL NEXT-DAY CLICK PREDICTION DATASET")
print("=" * 60)

print(f"📊 Task Information:")
print(f"   Objective: {dataset_info['task']}")
print(f"   Training period: {dataset_info['training_period']}")
print(f"   Test date: {dataset_info['test_date']}")
print(f"   Created: {dataset_info['creation_date'][:10]}")

print(f"\n📈 Dataset Statistics:")
print(f"   Total samples: {dataset_info['n_samples']:,}")
print(f"   Positive samples: {dataset_info['n_positive']:,} ({dataset_info['positive_rate']*100:.1f}%)")
print(f"   Negative samples: {dataset_info['n_negative']:,} ({(1-dataset_info['positive_rate'])*100:.1f}%)")
print(f"   Users analyzed: {dataset_info['n_users']:,}")
print(f"   News with embeddings: {dataset_info['n_news_with_embeddings']:,}")

print(f"\n🧠 Feature Structure:")
print(f"   User feature dimension: {dataset_info['user_feature_dim']}")
print(f"   News embedding dimension: {dataset_info['news_embedding_dim']}")
print(f"   Total combined dimension: {dataset_info['total_feature_dim']}")

breakdown = dataset_info['feature_breakdown']
print(f"\n🔧 Feature Breakdown:")
print(f"   Statistical features: {breakdown['statistical_features']} dims")
print(f"   User embedding: {breakdown['user_embedding']} dims (from clicked news entities)")
print(f"   History embedding: {breakdown['history_embedding']} dims (from browsing history entities)")
print(f"   News embedding: {breakdown['news_embedding']} dims (from news entities)")

print(f"\n✅ DKRL Integration:")
print(f"   ✓ Entity embeddings enhance news representation")
print(f"   ✓ User preferences learned from entity interactions") 
print(f"   ✓ Knowledge graph structure preserved in embeddings")
print(f"   ✓ Temporal split prevents data leakage")


🎯 DKRL NEXT-DAY CLICK PREDICTION DATASET
📊 Task Information:
   Objective: next_day_click_prediction
   Training period: 2019-11-09 to 2019-11-12
   Test date: 2019-11-13
   Created: 2025-07-26

📈 Dataset Statistics:
   Total samples: 18,309
   Positive samples: 759 (4.1%)
   Negative samples: 17,550 (95.9%)
   Users analyzed: 500
   News with embeddings: 51,282

🧠 Feature Structure:
   User feature dimension: 219
   News embedding dimension: 100
   Total combined dimension: 319

🔧 Feature Breakdown:
   Statistical features: 19 dims
   User embedding: 100 dims (from clicked news entities)
   History embedding: 100 dims (from browsing history entities)
   News embedding: 100 dims (from news entities)

✅ DKRL Integration:
   ✓ Entity embeddings enhance news representation
   ✓ User preferences learned from entity interactions
   ✓ Knowledge graph structure preserved in embeddings
   ✓ Temporal split prevents data leakage


In [6]:
# Data quality checks
print("🔍 Data Quality Assessment")
print("=" * 40)

# Check for missing values
user_missing = np.isnan(X_user).sum()
news_missing = np.isnan(X_news).sum()
label_missing = np.isnan(y).sum()

print(f"📊 Missing Values:")
print(f"   User features: {user_missing:,} missing values")
print(f"   News embeddings: {news_missing:,} missing values") 
print(f"   Labels: {label_missing:,} missing values")

# Check value ranges
print(f"\n📈 Value Ranges:")
print(f"   User features - Min: {X_user.min():.3f}, Max: {X_user.max():.3f}")
print(f"   News embeddings - Min: {X_news.min():.3f}, Max: {X_news.max():.3f}")
print(f"   Labels - Min: {y.min()}, Max: {y.max()}")

# Check data types and shapes consistency
print(f"\n🔧 Data Types & Consistency:")
print(f"   User features dtype: {X_user.dtype}")
print(f"   News embeddings dtype: {X_news.dtype}")
print(f"   Labels dtype: {y.dtype}")
print(f"   Shape consistency: {X_user.shape[0] == X_news.shape[0] == y.shape[0]}")

# Label distribution
unique_labels, counts = np.unique(y, return_counts=True)
print(f"\n🎯 Label Distribution:")
for label, count in zip(unique_labels, counts):
    print(f"   Label {label}: {count:,} samples ({count/len(y)*100:.1f}%)")


🔍 Data Quality Assessment
📊 Missing Values:
   User features: 0 missing values
   News embeddings: 0 missing values
   Labels: 0 missing values

📈 Value Ranges:
   User features - Min: -0.224, Max: 6216.000
   News embeddings - Min: -0.299, Max: 0.269
   Labels - Min: 0, Max: 1

🔧 Data Types & Consistency:
   User features dtype: float64
   News embeddings dtype: float64
   Labels dtype: int64
   Shape consistency: True

🎯 Label Distribution:
   Label 0: 17,550 samples (95.9%)
   Label 1: 759 samples (4.1%)


In [7]:
# Analyze feature components
print("🔬 Feature Component Analysis")
print("=" * 40)

# Extract feature components
X_stats = X_user[:, :19]  # Statistical features
X_user_emb = X_user[:, 19:119]  # User embedding
X_hist_emb = X_user[:, 119:219]  # History embedding

stat_feature_names = feature_names[:19]

print(f"📊 Statistical Features (first 19 dimensions):")
stats_df = pd.DataFrame(X_stats, columns=stat_feature_names)
print(stats_df.describe())

print(f"\n🧠 Embedding Statistics:")
print(f"   User embedding - Mean: {X_user_emb.mean():.4f}, Std: {X_user_emb.std():.4f}")
print(f"   History embedding - Mean: {X_hist_emb.mean():.4f}, Std: {X_hist_emb.std():.4f}")
print(f"   News embedding - Mean: {X_news.mean():.4f}, Std: {X_news.std():.4f}")

print(f"\n🎯 Feature Correlations with Target:")
correlations = []
for i, feature_name in enumerate(stat_feature_names):
    corr = np.corrcoef(X_stats[:, i], y)[0, 1]
    correlations.append((feature_name, corr))

# Sort by absolute correlation
correlations.sort(key=lambda x: abs(x[1]), reverse=True)

print("   Top 5 most correlated statistical features:")
for feature, corr in correlations[:5]:
    print(f"     {feature}: {corr:.4f}")

print("   Bottom 5 least correlated statistical features:")
for feature, corr in correlations[-5:]:
    print(f"     {feature}: {corr:.4f}")


🔬 Feature Component Analysis
📊 Statistical Features (first 19 dimensions):
       total_sessions  total_days_active  session_span_hours  \
count    18309.000000       18309.000000        18309.000000   
mean         6.619422           2.467202           42.752521   
std          5.161401           1.048776           28.770475   
min          1.000000           1.000000            0.000000   
25%          3.000000           2.000000           24.444444   
50%          5.000000           2.000000           32.963889   
75%          8.000000           3.000000           73.672778   
max         37.000000           4.000000           95.474722   

       total_history_items  total_impressions  total_clicks   overall_ctr  \
count         18309.000000       18309.000000  18309.000000  18309.000000   
mean            624.749140         299.762139     11.917363      0.049127   
std             970.728414         227.426193     10.679068      0.038599   
min               0.000000           2.0

In [8]:
# Analyze metadata patterns
print("📋 Metadata Analysis")
print("=" * 30)

print(f"👥 User Analysis:")
print(f"   Unique users: {metadata_df['user_id'].nunique():,}")
print(f"   Samples per user - Mean: {len(metadata_df) / metadata_df['user_id'].nunique():.1f}")
print(f"   User with most samples: {metadata_df['user_id'].value_counts().iloc[0]} samples")

print(f"\n📰 News Analysis:")
print(f"   Unique news articles: {metadata_df['news_id'].nunique():,}")
print(f"   Samples per news - Mean: {len(metadata_df) / metadata_df['news_id'].nunique():.1f}")
print(f"   Most shown news: {metadata_df['news_id'].value_counts().iloc[0]} times")

print(f"\n📅 Temporal Analysis:")
metadata_df['datetime'] = pd.to_datetime(metadata_df['datetime'])
print(f"   Date range: {metadata_df['datetime'].min().date()} to {metadata_df['datetime'].max().date()}")
print(f"   Time span: {(metadata_df['datetime'].max() - metadata_df['datetime'].min()).days} days")

print(f"\n🏷️ Category Analysis:")
category_counts = metadata_df['news_category'].value_counts()
print(f"   Unique categories: {len(category_counts)}")
print(f"   Top 5 categories:")
for category, count in category_counts.head().items():
    print(f"     {category}: {count:,} samples ({count/len(metadata_df)*100:.1f}%)")

print(f"\n🎯 Click Rate by Category:")
click_by_category = metadata_df.groupby('news_category')['click'].agg(['count', 'sum', 'mean']).round(3)
click_by_category['click_rate'] = click_by_category['mean']
click_by_category = click_by_category.sort_values('click_rate', ascending=False)
print(click_by_category.head())


📋 Metadata Analysis
👥 User Analysis:
   Unique users: 239
   Samples per user - Mean: 76.6
   User with most samples: 554 samples

📰 News Analysis:
   Unique news articles: 1,608
   Samples per news - Mean: 11.4
   Most shown news: 285 times

📅 Temporal Analysis:
   Date range: 2019-11-13 to 2019-11-13
   Time span: 0 days

🏷️ Category Analysis:
   Unique categories: 15
   Top 5 categories:
     news: 4,625 samples (25.3%)
     lifestyle: 2,761 samples (15.1%)
     finance: 1,768 samples (9.7%)
     sports: 1,431 samples (7.8%)
     foodanddrink: 1,347 samples (7.4%)

🎯 Click Rate by Category:
               count  sum   mean  click_rate
news_category                               
weather          258   25  0.097       0.097
video            299   22  0.074       0.074
music            744   45  0.060       0.060
lifestyle       2761  144  0.052       0.052
news            4625  221  0.048       0.048


In [9]:
# Analyze user profiles
print("👤 User Profile Analysis")
print("=" * 35)

# Convert user profiles to DataFrame for analysis
profile_data = []
for user_id, profile in user_profiles.items():
    profile_row = {
        'user_id': user_id,
        'total_sessions': profile['total_sessions'],
        'total_days_active': profile['total_days_active'],
        'total_clicks': profile['total_clicks'],
        'overall_ctr': profile['overall_ctr'],
        'category_diversity': profile['category_diversity'],
        'top_category': profile['top_category'],
        'embedding_coverage': profile['embedding_coverage'],
        'history_coverage': profile['history_coverage']
    }
    profile_data.append(profile_row)

profiles_df = pd.DataFrame(profile_data)

print(f"📊 User Profile Statistics:")
print(profiles_df[['total_sessions', 'total_days_active', 'total_clicks', 'overall_ctr', 'category_diversity']].describe())

print(f"\n🎯 CTR Distribution:")
ctr_bins = [0, 0.01, 0.05, 0.1, 0.2, 1.0]
ctr_labels = ['0-1%', '1-5%', '5-10%', '10-20%', '20%+']
profiles_df['ctr_bin'] = pd.cut(profiles_df['overall_ctr'], bins=ctr_bins, labels=ctr_labels)
print(profiles_df['ctr_bin'].value_counts())

print(f"\n📚 Top User Categories:")
print(profiles_df['top_category'].value_counts().head())

print(f"\n🔗 Embedding Coverage:")
print(f"   User embedding coverage - Mean: {profiles_df['embedding_coverage'].mean():.3f}")
print(f"   History embedding coverage - Mean: {profiles_df['history_coverage'].mean():.3f}")

# Show example profiles
print(f"\n👤 Sample User Profiles:")
sample_users = profiles_df.sample(3)
for _, user in sample_users.iterrows():
    print(f"\n   User {user['user_id']}:")
    print(f"     Sessions: {user['total_sessions']}, Clicks: {user['total_clicks']}, CTR: {user['overall_ctr']:.3f}")
    print(f"     Top category: {user['top_category']}, Diversity: {user['category_diversity']}")
    print(f"     Embedding coverage: {user['embedding_coverage']:.3f}")


👤 User Profile Analysis
📊 User Profile Statistics:
       total_sessions  total_days_active  total_clicks  overall_ctr  \
count      500.000000          500.00000    500.000000   500.000000   
mean         4.318000            2.13600      6.740000     0.077919   
std          3.956284            0.99372      7.240528     0.083747   
min          1.000000            1.00000      1.000000     0.005236   
25%          2.000000            1.00000      2.000000     0.029808   
50%          3.000000            2.00000      4.000000     0.050203   
75%          5.250000            3.00000      8.000000     0.090909   
max         37.000000            4.00000     50.000000     0.500000   

       category_diversity  
count          500.000000  
mean             3.558000  
std              2.215503  
min              1.000000  
25%              2.000000  
50%              3.000000  
75%              5.000000  
max             14.000000  

🎯 CTR Distribution:
ctr_bin
1-5%      243
5-10%     143


In [10]:
# Final summary and usage recommendations
print("🎯 DATASET READY FOR DKRL EXPERIMENTS!")
print("=" * 50)

print("✅ Dataset Validation Complete:")
print(f"   ✓ {len(y):,} samples loaded successfully")
print(f"   ✓ No missing values detected")
print(f"   ✓ Consistent shapes across all components")
print(f"   ✓ Realistic click rate ({dataset_info['positive_rate']*100:.1f}%)")
print(f"   ✓ Rich feature set ({dataset_info['total_feature_dim']} dimensions)")

print(f"\n🧠 DKRL Features:")
print(f"   ✓ Entity embeddings integrated into news representation")
print(f"   ✓ User preferences modeled through entity interactions")
print(f"   ✓ Historical browsing patterns captured in embeddings")
print(f"   ✓ Knowledge graph structure preserved")

print(f"\n🚀 Ready for Training:")
print(f"   📊 X_user.shape: {X_user.shape}")
print(f"   📰 X_news.shape: {X_news.shape}")
print(f"   🎯 y.shape: {y.shape}")

print(f"\n💡 Usage Examples:")
print(f"   # Option 1: Combined features")
print(f"   X_combined = np.concatenate([X_user, X_news], axis=1)")
print(f"   # model.fit(X_combined, y)")
print(f"   ")
print(f"   # Option 2: Separate processing")
print(f"   # Statistical: X_user[:, :19]")
print(f"   # User embedding: X_user[:, 19:119]") 
print(f"   # History embedding: X_user[:, 119:219]")
print(f"   # News embedding: X_news")

print(f"\n🎉 Dataset analysis complete!")
print(f"   Your DKRL next-day click prediction dataset is ready for experimentation!")


🎯 DATASET READY FOR DKRL EXPERIMENTS!
✅ Dataset Validation Complete:
   ✓ 18,309 samples loaded successfully
   ✓ No missing values detected
   ✓ Consistent shapes across all components
   ✓ Realistic click rate (4.1%)
   ✓ Rich feature set (319 dimensions)

🧠 DKRL Features:
   ✓ Entity embeddings integrated into news representation
   ✓ User preferences modeled through entity interactions
   ✓ Historical browsing patterns captured in embeddings
   ✓ Knowledge graph structure preserved

🚀 Ready for Training:
   📊 X_user.shape: (18309, 219)
   📰 X_news.shape: (18309, 100)
   🎯 y.shape: (18309,)

💡 Usage Examples:
   # Option 1: Combined features
   X_combined = np.concatenate([X_user, X_news], axis=1)
   # model.fit(X_combined, y)
   
   # Option 2: Separate processing
   # Statistical: X_user[:, :19]
   # User embedding: X_user[:, 19:119]
   # History embedding: X_user[:, 119:219]
   # News embedding: X_news

🎉 Dataset analysis complete!
   Your DKRL next-day click prediction dataset i

In [14]:
# Import the DKRL functions
import sys
sys.path.append('../../../DKRL-module')
from dkrl import *

print("🧠 Applying DKRL to MIND Click Prediction Dataset")
print("=" * 60)

# Set DKRL parameters as specified
r = 15          # Rank parameter
penalty = 1/200 # Penalty = 0.01
tol = 1e-4      # Convergence tolerance
T = 10000        # Maximum iterations

print(f"📊 DKRL Parameters:")
print(f"   Rank (r): {r}")
print(f"   Penalty: {penalty}")
print(f"   Tolerance: {tol}")
print(f"   Max iterations: {T}")

# Data setup: X_news as Z, X_user as X
print(f"\n🔧 Data Setup:")
print(f"   Z (news embeddings): {X_news.shape} - Entity embeddings from knowledge graph")
print(f"   X (user features): {X_user.shape} - Behavioral stats + user/history embeddings")
print(f"   Target (CTR): {y.shape} - Binary click labels")
print(f"   Positive rate: {y.mean():.4f}")

# Normalize features for stable kernel computation
from sklearn.preprocessing import StandardScaler

print(f"\n⚖️  Normalizing features for kernel computation...")
scaler_Z = StandardScaler()
scaler_X = StandardScaler()

Z_norm = scaler_Z.fit_transform(X_news)    # Normalize news embeddings  
X_norm = scaler_X.fit_transform(X_user)    # Normalize user features

print(f"   News embeddings normalized: {Z_norm.shape}")
print(f"   User features normalized: {X_norm.shape}")

# Compute kernel matrices
print(f"\n🔄 Computing kernel matrices...")
N = len(y)
KZ = np.dot(Z_norm, Z_norm.T) / Z_norm.shape[1]  # News kernel (18309 x 18309)
KX = np.dot(X_norm, X_norm.T) / X_norm.shape[1]  # User kernel (18309 x 18309)

print(f"   KZ (news kernel): {KZ.shape}")
print(f"   KX (user kernel): {KX.shape}")

print("✅ Kernel matrices ready for DKRL!")


🧠 Applying DKRL to MIND Click Prediction Dataset
📊 DKRL Parameters:
   Rank (r): 15
   Penalty: 0.005
   Tolerance: 0.0001
   Max iterations: 10000

🔧 Data Setup:
   Z (news embeddings): (18309, 100) - Entity embeddings from knowledge graph
   X (user features): (18309, 219) - Behavioral stats + user/history embeddings
   Target (CTR): (18309,) - Binary click labels
   Positive rate: 0.0415

⚖️  Normalizing features for kernel computation...
   News embeddings normalized: (18309, 100)
   User features normalized: (18309, 219)

🔄 Computing kernel matrices...
   KZ (news kernel): (18309, 18309)
   KX (user kernel): (18309, 18309)
✅ Kernel matrices ready for DKRL!


In [20]:
# Run DKRL algorithm
print("🚀 Running DKRL Dual Kernel Learning...")
print("=" * 50)

import time
start_time = time.time()

# Set DKRL parameters as specified
r = 25          # Rank parameter
penalty = 1/np.sqrt(N) # Penalty = 0.01
tol = 1e-4      # Convergence tolerance
T = 10000        # Maximum iterations

# Run DKRL with news embeddings as Z and user features as X
try:
    # Suppress warnings during optimization
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning)
        
        U_matrix, V_matrix, y_pred_dkrl = DKRL_dual(
            Z = Z_norm,              # News kernel (from embeddings)
            X = X_norm,              # User kernel (from features)
            y=y.astype(float),  # Convert to float for DKRL
            r=r,
            penalty=penalty,
            tol=tol,
            T=T
        )
    
    end_time = time.time()
    training_time = end_time - start_time
    
    print(f"\n✅ DKRL training completed!")
    print(f"   Training time: {training_time:.2f} seconds")
    print(f"   U matrix shape: {U_matrix.shape}")
    print(f"   V matrix shape: {V_matrix.shape}")
    print(f"   Predictions shape: {y_pred_dkrl.shape}")
    
    # Basic prediction statistics
    print(f"\n📊 Prediction Statistics:")
    print(f"   Actual CTR - Mean: {y.mean():.4f}, Std: {y.std():.4f}")
    print(f"   DKRL pred - Mean: {y_pred_dkrl.mean():.4f}, Std: {y_pred_dkrl.std():.4f}")
    print(f"   Prediction range: [{y_pred_dkrl.min():.4f}, {y_pred_dkrl.max():.4f}]")
    
    # Correlation with actual labels
    correlation = np.corrcoef(y, y_pred_dkrl)[0, 1]
    print(f"   Correlation with actual: {correlation:.4f}")
    
    # RMSE
    rmse = np.sqrt(np.mean((y - y_pred_dkrl) ** 2))
    print(f"   RMSE: {rmse:.6f}")
    
    success = True
    
except Exception as e:
    print(f"❌ Error during DKRL training: {e}")
    print(f"   This might be due to numerical instability")
    success = False
    raise

print(f"\n🎯 DKRL dual kernel learning {'completed successfully!' if success else 'encountered issues.'}")


🚀 Running DKRL Dual Kernel Learning...


  8%|▊         | 809/10000 [03:49<43:26,  3.53it/s]


✅ DKRL training completed!
   Training time: 229.44 seconds
   U matrix shape: (100, 25)
   V matrix shape: (219, 25)
   Predictions shape: (18309,)

📊 Prediction Statistics:
   Actual CTR - Mean: 0.0415, Std: 0.1993
   DKRL pred - Mean: 0.0142, Std: 0.1759
   Prediction range: [-1.8823, 2.3156]
   Correlation with actual: 0.3363
   RMSE: 0.218685

🎯 DKRL dual kernel learning completed successfully!


In [ ]:
## 8. User-Level Aggregation

print("🔄 Aggregating Dataset to User Level")
print("=" * 50)

# Current data is at impression level (user-news pairs)
# We need to aggregate to user level: one row per user with avg embeddings and CTR

print(f"📊 Current Data Structure:")
print(f"   Impressions (user-news pairs): {len(y):,}")
print(f"   Unique users: {metadata_df['user_id'].nunique():,}")
print(f"   Avg impressions per user: {len(y) / metadata_df['user_id'].nunique():.1f}")

# Group by user to calculate aggregates
print(f"\n🔄 Performing user-level aggregation...")

# Get user-level statistics
user_stats = metadata_df.groupby('user_id').agg({
    'click': ['count', 'sum', 'mean'],  # total impressions, total clicks, CTR
    'news_id': 'nunique'  # unique news articles shown
}).round(4)

# Flatten column names
user_stats.columns = ['total_impressions', 'total_clicks', 'ctr', 'unique_news']
user_stats = user_stats.reset_index()

print(f"✓ Calculated user-level statistics for {len(user_stats)} users")

# Create mapping from impression index to user_id
impression_to_user = dict(zip(range(len(metadata_df)), metadata_df['user_id']))

# Aggregate user features (should be same for each user, so take first occurrence)
print(f"🧠 Aggregating user features...")
user_features_agg = []
user_ids_ordered = []

for user_id in user_stats['user_id']:
    # Find first occurrence of this user in the impression data
    user_indices = metadata_df[metadata_df['user_id'] == user_id].index
    first_idx = user_indices[0]
    
    # Take user features from first occurrence (should be same for all)
    user_features_agg.append(X_user[first_idx])
    user_ids_ordered.append(user_id)

X_user_agg = np.array(user_features_agg)
print(f"✓ User features aggregated: {X_user_agg.shape}")

# Aggregate news embeddings (average across all impressions for each user)
print(f"📰 Aggregating news embeddings (average per user)...")
news_embeddings_agg = []

for user_id in user_stats['user_id']:
    # Find all impressions for this user
    user_indices = metadata_df[metadata_df['user_id'] == user_id].index
    
    # Average news embeddings across all impressions for this user
    user_news_embeddings = X_news[user_indices]
    avg_news_embedding = user_news_embeddings.mean(axis=0)
    news_embeddings_agg.append(avg_news_embedding)

X_news_agg = np.array(news_embeddings_agg)
print(f"✓ News embeddings aggregated: {X_news_agg.shape}")

# Create target variable (CTR per user)
y_agg = user_stats['ctr'].values
print(f"✓ Target variable (CTR): {y_agg.shape}")

# Verify data integrity
print(f"\n🔍 Data Integrity Check:")
print(f"   All arrays same length: {len(X_user_agg) == len(X_news_agg) == len(y_agg)}")
print(f"   User IDs match: {len(user_ids_ordered) == len(user_stats)}")

# Summary statistics
print(f"\n📈 Aggregated Dataset Summary:")
print(f"   Total users: {len(y_agg):,}")
print(f"   User features dimension: {X_user_agg.shape[1]}")
print(f"   News embedding dimension: {X_news_agg.shape[1]}")
print(f"   CTR statistics:")
print(f"     Mean: {y_agg.mean():.4f}")
print(f"     Std: {y_agg.std():.4f}")
print(f"     Min: {y_agg.min():.4f}")
print(f"     Max: {y_agg.max():.4f}")
print(f"     Median: {np.median(y_agg):.4f}")

# Additional user-level statistics
print(f"\n📊 User Behavior Statistics:")
print(f"   Impressions per user:")
print(f"     Mean: {user_stats['total_impressions'].mean():.1f}")
print(f"     Std: {user_stats['total_impressions'].std():.1f}")
print(f"     Min: {user_stats['total_impressions'].min()}")
print(f"     Max: {user_stats['total_impressions'].max()}")

print(f"   Clicks per user:")
print(f"     Mean: {user_stats['total_clicks'].mean():.1f}")
print(f"     Std: {user_stats['total_clicks'].std():.1f}")
print(f"     Min: {user_stats['total_clicks'].min()}")
print(f"     Max: {user_stats['total_clicks'].max()}")

print(f"   Unique news per user:")
print(f"     Mean: {user_stats['unique_news'].mean():.1f}")
print(f"     Std: {user_stats['unique_news'].std():.1f}")
print(f"     Min: {user_stats['unique_news'].min()}")
print(f"     Max: {user_stats['unique_news'].max()}")

print(f"\n✅ User-level aggregation complete!")
print(f"   Ready for user-level modeling with CTR as continuous target")


🔄 Aggregating Dataset to User Level
📊 Current Data Structure:
   Impressions (user-news pairs): 18,309
   Unique users: 239
   Avg impressions per user: 76.6

🔄 Performing user-level aggregation...
✓ Calculated user-level statistics for 239 users
🧠 Aggregating user features...
✓ User features aggregated: (239, 219)
📰 Aggregating news embeddings (average per user)...
✓ News embeddings aggregated: (239, 100)
✓ Target variable (CTR): (239,)

🔍 Data Integrity Check:
   All arrays same length: True
   User IDs match: True

📈 Aggregated Dataset Summary:
   Total users: 239
   User features dimension: 219
   News embedding dimension: 100
   CTR statistics:
     Mean: 0.0785
     Std: 0.0868
     Min: 0.0071
     Max: 0.5000
     Median: 0.0513

📊 User Behavior Statistics:
   Impressions per user:
     Mean: 76.6
     Std: 84.9
     Min: 2
     Max: 554
   Clicks per user:
     Mean: 3.2
     Std: 2.9
     Min: 1
     Max: 18
   Unique news per user:
     Mean: 62.9
     Std: 59.1
     Min: 2


In [20]:
# Import the DKRL functions
import sys
sys.path.append('../../../DKRL-module')
from dkrl import *

print("🧠 Applying DKRL to MIND Click Prediction Dataset")
print("=" * 60)

# Set DKRL parameters as specified
r = 15          # Rank parameter
penalty = 1/200 # Penalty = 0.01
tol = 1e-4      # Convergence tolerance
T = 10000        # Maximum iterations

print(f"📊 DKRL Parameters:")
print(f"   Rank (r): {r}")
print(f"   Penalty: {penalty}")
print(f"   Tolerance: {tol}")
print(f"   Max iterations: {T}")

# Data setup: X_news as Z, X_user as X
print(f"\n🔧 Data Setup:")
print(f"   Z (news embeddings): {X_news.shape} - Entity embeddings from knowledge graph")
print(f"   X (user features): {X_user.shape} - Behavioral stats + user/history embeddings")
print(f"   Target (CTR): {y.shape} - Binary click labels")
print(f"   Positive rate: {y.mean():.4f}")

# Normalize features for stable kernel computation
from sklearn.preprocessing import StandardScaler

print(f"\n⚖️  Normalizing features for kernel computation...")
scaler_Z = StandardScaler()
scaler_X = StandardScaler()

Z_agg_norm = scaler_Z.fit_transform(X_news_agg)    # Normalize news embeddings  
X_agg_norm = scaler_X.fit_transform(X_user_agg)    # Normalize user features

print(f"   News embeddings normalized: {Z_agg_norm.shape}")
print(f"   User features normalized: {X_agg_norm.shape}")

# Compute kernel matrices
print(f"\n🔄 Computing kernel matrices...")
N = len(y)
KZ = np.dot(Z_agg_norm, Z_agg_norm.T) / Z_agg_norm.shape[1]  # News kernel (18309 x 18309)
KX = np.dot(X_agg_norm, X_agg_norm.T) / X_agg_norm.shape[1]  # User kernel (18309 x 18309)

print(f"   KZ (news kernel): {KZ.shape}")
print(f"   KX (user kernel): {KX.shape}")

print("✅ Kernel matrices ready for DKRL!")

🧠 Applying DKRL to MIND Click Prediction Dataset
📊 DKRL Parameters:
   Rank (r): 15
   Penalty: 0.005
   Tolerance: 0.0001
   Max iterations: 10000

🔧 Data Setup:
   Z (news embeddings): (18309, 100) - Entity embeddings from knowledge graph
   X (user features): (18309, 219) - Behavioral stats + user/history embeddings
   Target (CTR): (18309,) - Binary click labels
   Positive rate: 0.0415

⚖️  Normalizing features for kernel computation...
   News embeddings normalized: (239, 100)
   User features normalized: (239, 219)

🔄 Computing kernel matrices...
   KZ (news kernel): (239, 239)
   KX (user kernel): (239, 239)
✅ Kernel matrices ready for DKRL!


In [ ]:
 ###### DKRL analysis ######
 # Run DKRL algorithm
print("🚀 Running DKRL Dual Kernel Learning...")
print("=" * 50)

import time
start_time = time.time()

# Set DKRL parameters as specified
r = 1     # Rank parameter
penalty = 1/np.sqrt(N) # Penalty = 0.01
tol = 1e-4      # Convergence tolerance
T = 10000        # Maximum iterations

# Run DKRL with news embeddings as Z and user features as X
try:
    # Suppress warnings during optimization
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning)
        
        U_matrix, V_matrix, y_pred_dkrl = DKRL_dual(
            Z = Z_agg_norm,              # News kernel (from embeddings)
            X = X_agg_norm,              # User kernel (from features)
            y=y_agg.astype(float),  # Convert to float for DKRL
            r=r,
            penalty=penalty,
            tol=tol,
            T=T
        )
    
    end_time = time.time()
    training_time = end_time - start_time
    
    print(f"\n✅ DKRL training completed!")
    print(f"   Training time: {training_time:.2f} seconds")
    print(f"   U matrix shape: {U_matrix.shape}")
    print(f"   V matrix shape: {V_matrix.shape}")
    print(f"   Predictions shape: {y_pred_dkrl.shape}")
    
    # Basic prediction statistics
    print(f"\n📊 Prediction Statistics:")
    print(f"   Actual CTR - Mean: {y.mean():.4f}, Std: {y.std():.4f}")
    print(f"   DKRL pred - Mean: {y_pred_dkrl.mean():.4f}, Std: {y_pred_dkrl.std():.4f}")
    print(f"   Prediction range: [{y_pred_dkrl.min():.4f}, {y_pred_dkrl.max():.4f}]")
    
    # Correlation with actual labels
    correlation = np.corrcoef(y_agg, y_pred_dkrl)[0, 1]
    print(f"   Correlation with actual: {correlation:.4f}")
    
    # RMSE
    rmse = np.sqrt(np.mean((y_agg - y_pred_dkrl) ** 2))
    print(f"   RMSE: {rmse:.6f}")
    
    success = True
    
except Exception as e:
    print(f"❌ Error during DKRL training: {e}")
    print(f"   This might be due to numerical instability")
    success = False
    raise

print(f"\n🎯 DKRL dual kernel learning {'completed successfully!' if success else 'encountered issues.'}")

🚀 Running DKRL Dual Kernel Learning...


  5%|▍         | 461/10000 [00:00<00:03, 2669.32it/s]


✅ DKRL training completed!
   Training time: 0.18 seconds
   U matrix shape: (100, 1)
   V matrix shape: (219, 1)
   Predictions shape: (239,)

📊 Prediction Statistics:
   Actual CTR - Mean: 0.0415, Std: 0.1993
   DKRL pred - Mean: 0.0785, Std: 0.0868
   Prediction range: [0.0069, 0.5000]
   Correlation with actual: 1.0000
   RMSE: 0.000161

🎯 DKRL dual kernel learning completed successfully!


In [31]:
## 8. Session-Level Aggregation

print("🔄 Aggregating Dataset to Session Level")
print("=" * 60)

# Current data is at impression level (user-news pairs)
# We need to aggregate to session level: one row per session with avg embeddings and CTR

print(f"📊 Current Data Structure:")
print(f"   Impressions (user-news pairs): {len(y):,}")
print(f"   Unique users: {metadata_df['user_id'].nunique():,}")
print(f"   Avg impressions per user: {len(y) / metadata_df['user_id'].nunique():.1f}")

# We need to create session identifiers
# Since we don't have explicit session IDs, we'll create them based on user and temporal patterns
# Let's assume each user's impressions within a short time window constitute a session

print(f"\n🔄 Creating session identifiers...")

# Sort by user and datetime to identify sessions
metadata_sorted = metadata_df.sort_values(['user_id', 'datetime']).copy()

# Create session breaks based on time gaps > 30 minutes
time_diff = metadata_sorted.groupby('user_id')['datetime'].diff()
session_breaks = (time_diff > pd.Timedelta(minutes=30)) | time_diff.isna()

# Create session IDs
metadata_sorted['session_break'] = session_breaks
metadata_sorted['session_id'] = metadata_sorted.groupby('user_id')['session_break'].cumsum()
metadata_sorted['user_session_id'] = metadata_sorted['user_id'] + '_S' + metadata_sorted['session_id'].astype(str)

# Merge back to original order
metadata_df = metadata_df.merge(
    metadata_sorted[['user_id', 'news_id', 'datetime', 'user_session_id']], 
    on=['user_id', 'news_id', 'datetime'], 
    how='left'
)

print(f"✓ Created session identifiers")
print(f"   Total sessions: {metadata_df['user_session_id'].nunique():,}")
print(f"   Avg impressions per session: {len(metadata_df) / metadata_df['user_session_id'].nunique():.1f}")

# Group by session to calculate aggregates
print(f"\n🔄 Performing session-level aggregation...")

# Get session-level statistics
session_stats = metadata_df.groupby('user_session_id').agg({
    'click': ['count', 'sum', 'mean'],  # total impressions, total clicks, CTR per session
    'news_id': 'nunique',  # unique news articles shown in session
    'user_id': 'first',    # user ID (should be same for all in session)
    'datetime': ['min', 'max']  # session start and end times
}).round(4)

# Flatten column names
session_stats.columns = ['total_impressions', 'total_clicks', 'session_ctr', 'unique_news', 'user_id', 'session_start', 'session_end']
session_stats = session_stats.reset_index()

# Calculate session duration in minutes
session_stats['session_duration_min'] = (session_stats['session_end'] - session_stats['session_start']).dt.total_seconds() / 60

print(f"✓ Calculated session-level statistics for {len(session_stats)} sessions")

# Aggregate user features (take first occurrence for each session)
print(f"🧠 Aggregating user features per session...")
session_features_agg = []
session_ids_ordered = []

for session_id in session_stats['user_session_id']:
    # Find first occurrence of this session in the impression data
    session_indices = metadata_df[metadata_df['user_session_id'] == session_id].index
    first_idx = session_indices[0]
    
    # Take user features from first occurrence (should be same within session)
    session_features_agg.append(X_user[first_idx])
    session_ids_ordered.append(session_id)

X_user_sess = np.array(session_features_agg)
print(f"✓ User features aggregated: {X_user_sess.shape}")

# Aggregate news embeddings (average across all impressions in each session)
print(f"📰 Aggregating news embeddings (average per session)...")
news_embeddings_sess = []

for session_id in session_stats['user_session_id']:
    # Find all impressions for this session
    session_indices = metadata_df[metadata_df['user_session_id'] == session_id].index
    
    # Average news embeddings across all impressions in this session
    session_news_embeddings = X_news[session_indices]
    avg_news_embedding = session_news_embeddings.mean(axis=0)
    news_embeddings_sess.append(avg_news_embedding)

X_news_sess = np.array(news_embeddings_sess)
print(f"✓ News embeddings aggregated: {X_news_sess.shape}")

# Create target variable (CTR per session)
y_sess = session_stats['session_ctr'].values
print(f"✓ Target variable (Session CTR): {y_sess.shape}")

# Verify data integrity
print(f"\n🔍 Data Integrity Check:")
print(f"   All arrays same length: {len(X_user_sess) == len(X_news_sess) == len(y_sess)}")
print(f"   Session IDs match: {len(session_ids_ordered) == len(session_stats)}")

# Summary statistics
print(f"\n📈 Session-Level Dataset Summary:")
print(f"   Total sessions: {len(y_sess):,}")
print(f"   User features dimension: {X_user_sess.shape[1]}")
print(f"   News embedding dimension: {X_news_sess.shape[1]}")
print(f"   Session CTR statistics:")
print(f"     Mean: {y_sess.mean():.4f}")
print(f"     Std: {y_sess.std():.4f}")
print(f"     Min: {y_sess.min():.4f}")
print(f"     Max: {y_sess.max():.4f}")
print(f"     Median: {np.median(y_sess):.4f}")

# Additional session-level statistics
print(f"\n📊 Session Behavior Statistics:")
print(f"   Impressions per session:")
print(f"     Mean: {session_stats['total_impressions'].mean():.1f}")
print(f"     Std: {session_stats['total_impressions'].std():.1f}")
print(f"     Min: {session_stats['total_impressions'].min()}")
print(f"     Max: {session_stats['total_impressions'].max()}")

print(f"   Clicks per session:")
print(f"     Mean: {session_stats['total_clicks'].mean():.1f}")
print(f"     Std: {session_stats['total_clicks'].std():.1f}")
print(f"     Min: {session_stats['total_clicks'].min()}")
print(f"     Max: {session_stats['total_clicks'].max()}")

print(f"   Session duration:")
print(f"     Mean: {session_stats['session_duration_min'].mean():.1f} minutes")
print(f"     Std: {session_stats['session_duration_min'].std():.1f} minutes")
print(f"     Min: {session_stats['session_duration_min'].min():.1f} minutes")
print(f"     Max: {session_stats['session_duration_min'].max():.1f} minutes")

print(f"   Unique news per session:")
print(f"     Mean: {session_stats['unique_news'].mean():.1f}")
print(f"     Std: {session_stats['unique_news'].std():.1f}")
print(f"     Min: {session_stats['unique_news'].min()}")
print(f"     Max: {session_stats['unique_news'].max()}")

# Sessions per user analysis
sessions_per_user = session_stats.groupby('user_id').size()
print(f"\n👥 Sessions per User:")
print(f"     Mean: {sessions_per_user.mean():.1f}")
print(f"     Std: {sessions_per_user.std():.1f}")
print(f"     Min: {sessions_per_user.min()}")
print(f"     Max: {sessions_per_user.max()}")

print(f"\n✅ Session-level aggregation complete!")
print(f"   Ready for session-level modeling with CTR as continuous target")
print(f"   Sessions capture browsing behavior within focused time windows")
print(f"   Each session represents a coherent user interaction period")


🔄 Aggregating Dataset to Session Level
📊 Current Data Structure:
   Impressions (user-news pairs): 18,309
   Unique users: 239
   Avg impressions per user: 76.6

🔄 Creating session identifiers...
✓ Created session identifiers
   Total sessions: 427
   Avg impressions per session: 42.9

🔄 Performing session-level aggregation...
✓ Calculated session-level statistics for 427 sessions
🧠 Aggregating user features per session...
✓ User features aggregated: (427, 219)
📰 Aggregating news embeddings (average per session)...
✓ News embeddings aggregated: (427, 100)
✓ Target variable (Session CTR): (427,)

🔍 Data Integrity Check:
   All arrays same length: True
   Session IDs match: True

📈 Session-Level Dataset Summary:
   Total sessions: 427
   User features dimension: 219
   News embedding dimension: 100
   Session CTR statistics:
     Mean: 0.0995
     Std: 0.1209
     Min: 0.0040
     Max: 0.5000
     Median: 0.0541

📊 Session Behavior Statistics:
   Impressions per session:
     Mean: 42.9


In [34]:
# Import the DKRL functions
import sys
sys.path.append('../../../DKRL-module')
from dkrl import *

print("🧠 Applying DKRL to MIND Click Prediction Dataset")
print("=" * 60)

# Set DKRL parameters as specified
r = 15          # Rank parameter
penalty = 1/200 # Penalty = 0.01
tol = 1e-4      # Convergence tolerance
T = 10000        # Maximum iterations

print(f"📊 DKRL Parameters:")
print(f"   Rank (r): {r}")
print(f"   Penalty: {penalty}")
print(f"   Tolerance: {tol}")
print(f"   Max iterations: {T}")

# Data setup: X_news as Z, X_user as X
print(f"\n🔧 Data Setup:")
print(f"   Z (news embeddings): {X_news.shape} - Entity embeddings from knowledge graph")
print(f"   X (user features): {X_user.shape} - Behavioral stats + user/history embeddings")
print(f"   Target (CTR): {y.shape} - Binary click labels")
print(f"   Positive rate: {y.mean():.4f}")

# Normalize features for stable kernel computation
from sklearn.preprocessing import StandardScaler

print(f"\n⚖️  Normalizing features for kernel computation...")
scaler_Z = StandardScaler()
scaler_X = StandardScaler()

Z_sess_norm = scaler_Z.fit_transform(X_news_sess)    # Normalize news embeddings  
X_sess_norm = scaler_X.fit_transform(X_user_sess)    # Normalize user features

print(f"   News embeddings normalized: {Z_sess_norm.shape}")
print(f"   User features normalized: {X_sess_norm.shape}")

# Compute kernel matrices
print(f"\n🔄 Computing kernel matrices...")
N = len(y)
KZ = np.dot(Z_sess_norm, Z_sess_norm.T) / Z_sess_norm.shape[1]  # News kernel (18309 x 18309)
KX = np.dot(X_sess_norm, X_sess_norm.T) / X_sess_norm.shape[1]  # User kernel (18309 x 18309)

print(f"   KZ (news kernel): {KZ.shape}")
print(f"   KX (user kernel): {KX.shape}")

print("✅ Kernel matrices ready for DKRL!")

🧠 Applying DKRL to MIND Click Prediction Dataset
📊 DKRL Parameters:
   Rank (r): 15
   Penalty: 0.005
   Tolerance: 0.0001
   Max iterations: 10000

🔧 Data Setup:
   Z (news embeddings): (18309, 100) - Entity embeddings from knowledge graph
   X (user features): (18309, 219) - Behavioral stats + user/history embeddings
   Target (CTR): (18309,) - Binary click labels
   Positive rate: 0.0415

⚖️  Normalizing features for kernel computation...
   News embeddings normalized: (427, 100)
   User features normalized: (427, 219)

🔄 Computing kernel matrices...
   KZ (news kernel): (427, 427)
   KX (user kernel): (427, 427)
✅ Kernel matrices ready for DKRL!


In [35]:
###### DKRL analysis ######
 # Run DKRL algorithm
print("🚀 Running DKRL Dual Kernel Learning...")
print("=" * 50)

import time
start_time = time.time()

# Set DKRL parameters as specified
r = 1     # Rank parameter
penalty = 1/np.sqrt(y_sess.shape[0]) # Penalty = 0.01
tol = 1e-4      # Convergence tolerance
T = 10000        # Maximum iterations

# Run DKRL with news embeddings as Z and user features as X
try:
    # Suppress warnings during optimization
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning)
        
        U_matrix, V_matrix, y_pred_dkrl = DKRL_dual(
            Z = Z_sess_norm,              # News kernel (from embeddings)
            X = X_sess_norm,              # User kernel (from features)
            y=y_sess.astype(float),  # Convert to float for DKRL
            r=r,
            penalty=penalty,
            tol=tol,
            T=T
        )
    
    end_time = time.time()
    training_time = end_time - start_time
    
    print(f"\n✅ DKRL training completed!")
    print(f"   Training time: {training_time:.2f} seconds")
    print(f"   U matrix shape: {U_matrix.shape}")
    print(f"   V matrix shape: {V_matrix.shape}")
    print(f"   Predictions shape: {y_pred_dkrl.shape}")
    
    # Basic prediction statistics
    print(f"\n📊 Prediction Statistics:")
    print(f"   Actual CTR - Mean: {y.mean():.4f}, Std: {y.std():.4f}")
    print(f"   DKRL pred - Mean: {y_pred_dkrl.mean():.4f}, Std: {y_pred_dkrl.std():.4f}")
    print(f"   Prediction range: [{y_pred_dkrl.min():.4f}, {y_pred_dkrl.max():.4f}]")
    
    # Correlation with actual labels
    correlation = np.corrcoef(y_sess, y_pred_dkrl)[0, 1]
    print(f"   Correlation with actual: {correlation:.4f}")
    
    # RMSE
    rmse = np.sqrt(np.mean((y_sess - y_pred_dkrl) ** 2))
    print(f"   RMSE: {rmse:.6f}")
    
    success = True
    
except Exception as e:
    print(f"❌ Error during DKRL training: {e}")
    print(f"   This might be due to numerical instability")
    success = False
    raise

print(f"\n🎯 DKRL dual kernel learning {'completed successfully!' if success else 'encountered issues.'}")

🚀 Running DKRL Dual Kernel Learning...


 70%|██████▉   | 6957/10000 [00:02<00:01, 2346.47it/s]


✅ DKRL training completed!
   Training time: 2.97 seconds
   U matrix shape: (100, 1)
   V matrix shape: (219, 1)
   Predictions shape: (427,)

📊 Prediction Statistics:
   Actual CTR - Mean: 0.0415, Std: 0.1993
   DKRL pred - Mean: 0.0807, Std: 0.1218
   Prediction range: [-0.0727, 0.5374]
   Correlation with actual: 0.9046
   RMSE: 0.056268

🎯 DKRL dual kernel learning completed successfully!


In [ ]:
# build several baseline models